# 02 — Vector Add（向量加法 & 融合算子）

## 背景

元素级（element-wise）运算是神经网络中最常见的操作之一：激活函数、残差加法、Dropout 等都属于此类。在 GPU 上，元素级运算是典型的**内存带宽瓶颈**（Memory-Bound）操作——计算极少，数据搬运量大。

**算子融合（Kernel Fusion）** 是提升性能的关键手段：将多个独立的元素级操作合并为单个 GPU 内核，避免多次全局内存读写。例如将 `A * B` 与 `ReLU` 融合为一个内核，比分别执行可节省大量内存带宽。

## 问题定义

**2-a：向量加法**
$$C[i] = A[i] + B[i]$$

**2-b：融合乘法 + ReLU（含寄存器缓冲优化）**
$$C[i] = \max\bigl(0,\ A[i] \times B[i]\bigr)$$

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
N = 1024 * 4096
A = torch.randn(N, dtype=torch.float16, device="cuda")
B = torch.randn(N, dtype=torch.float16, device="cuda")

def ref_add(A, B):       return A + B
def ref_mul_relu(A, B):  return torch.relu(A * B)

print(f"N = {N:,}, dtype = float16")
print("向量加法  :", ref_add(A, B).shape)
print("乘法+ReLU:", ref_mul_relu(A, B).shape)

## Triton 实现

Triton 内核中用 `tl.maximum(x, 0.0)` 实现 ReLU（比 `tl.where` 更简洁），Triton 会自动生成无分支的向量化指令。

In [ ]:
@triton.jit
def _add_kernel(A_ptr, B_ptr, C_ptr, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    c = tl.load(A_ptr + offs, mask=mask) + tl.load(B_ptr + offs, mask=mask)
    tl.store(C_ptr + offs, c, mask=mask)

@triton.jit
def _mul_relu_kernel(A_ptr, B_ptr, C_ptr, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    a = tl.load(A_ptr + offs, mask=mask)
    b = tl.load(B_ptr + offs, mask=mask)
    tl.store(C_ptr + offs, tl.maximum(a * b, 0.0), mask=mask)

BLOCK = 1024

def triton_add(A, B):
    C = torch.empty_like(A)
    _add_kernel[(triton.cdiv(A.numel(), BLOCK),)](A, B, C, A.numel(), BLOCK=BLOCK)
    return C

def triton_mul_relu(A, B):
    C = torch.empty_like(A)
    _mul_relu_kernel[(triton.cdiv(A.numel(), BLOCK),)](A, B, C, A.numel(), BLOCK=BLOCK)
    return C

print("向量加法  :", "✅" if torch.allclose(ref_add(A,B),       triton_add(A,B),      atol=1e-2) else "❌")
print("乘法+ReLU:", "✅" if torch.allclose(ref_mul_relu(A,B), triton_mul_relu(A,B), atol=1e-2) else "❌")

## TileLang 实现

TileLang 中 `T.if_then_else` 实现无分支 ReLU；`T.alloc_fragment` 分配寄存器缓冲，将全局内存读写次数降至最低（只读一次 A、B，只写一次 C）。

In [ ]:
BN = 1024

@tilelang.jit
def tl_add(A, B, BLOCK_N: int):
    N = T.const("N")
    A: T.Tensor((N,), T.float16);  B: T.Tensor((N,), T.float16)
    C = T.empty((N,), T.float16)
    with T.Kernel(N // BLOCK_N, threads=256) as bx:
        base = bx * BLOCK_N
        for i in T.Parallel(BLOCK_N):
            C[base + i] = A[base + i] + B[base + i]
    return C

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_mul_relu(A, B, BLOCK_N: int):
    N = T.const("N")
    dtype = T.float16
    A: T.Tensor((N,), dtype);  B: T.Tensor((N,), dtype)
    C = T.empty((N,), dtype)
    with T.Kernel(N // BLOCK_N, threads=256) as bx:
        A_r = T.alloc_fragment((BLOCK_N,), dtype)
        B_r = T.alloc_fragment((BLOCK_N,), dtype)
        C_r = T.alloc_fragment((BLOCK_N,), dtype)
        T.copy(A[bx * BLOCK_N:(bx + 1) * BLOCK_N], A_r)
        T.copy(B[bx * BLOCK_N:(bx + 1) * BLOCK_N], B_r)
        for i in T.Parallel(BLOCK_N):
            val = A_r[i] * B_r[i]
            C_r[i] = T.if_then_else(val > dtype(0), val, dtype(0))
        T.copy(C_r, C[bx * BLOCK_N:(bx + 1) * BLOCK_N])
    return C

k_add = tl_add.compile(N=N, BLOCK_N=BN)
k_mul = tl_mul_relu.compile(N=N, BLOCK_N=BN)

print("向量加法  :", "✅" if torch.allclose(ref_add(A,B),       k_add(A.clone(), B.clone()), atol=1e-2) else "❌")
print("乘法+ReLU:", "✅" if torch.allclose(ref_mul_relu(A,B), k_mul(A.clone(), B.clone()), atol=1e-2) else "❌")

## 性能对比

以**乘法 + ReLU** 为主进行三方对比（该算子融合效果最明显）。

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch\n(A+B)": bench(ref_add, [A, B]),
    "Triton\n(A+B)": bench(triton_add, [A, B]),
    "TileLang\n(A+B)": bench(k_add, [A, B]),
    "PyTorch\nMul+ReLU": bench(ref_mul_relu, [A, B]),
    "Triton\nMul+ReLU": bench(triton_mul_relu, [A, B]),
    "TileLang\nMul+ReLU": bench(k_mul, [A, B]),
}

for name, ms in results.items():
    print(f"  {name.replace(chr(10),' '):22s}: {ms:.4f} ms")

colors = ["#4C72B0", "#55A868", "#C44E52",
          "#6EB5FF", "#8FD9A8", "#FF7C7C"]
labels = list(results.keys())
values = list(results.values())

fig, ax = plt.subplots(figsize=(11, 4))
x = range(len(labels))
bars = ax.bar(x, values, color=colors, width=0.6, edgecolor="white", linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=9)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8)
# 分组分隔线
ax.axvline(2.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(1, max(values)*1.15, "向量加法", ha="center", fontsize=10, color="gray")
ax.text(4, max(values)*1.15, "乘法 + ReLU", ha="center", fontsize=10, color="gray")
ax.set_ylabel("延迟 (ms)")
ax.set_title(f"Vector Add & Mul+ReLU 性能对比  (N={N:,}, float16)")
ax.set_ylim(0, max(values)*1.25)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()